# OPT-03 Variability and uncertainty in regionalised LCIA

Day 1 focused on uncertainty attached to inventories and exchanges. Here, we focus instead on variability and uncertainty in the **characterization factors themselves**.


## Learning goals

After this notebook, you should be able to:

- distinguish Day 1 inventory uncertainty from Day 2 CF variability
- inspect how uncertainty is encoded in local `edges` JSON methods
- apply simple uncertain LCIA methods to a real BAFU activity
- run stochastic `EdgeLCIA` calculations with `use_distributions=True`
- run a Monte Carlo with a real regionalized method such as `AWARE 2.0`
- summarize score spread with direct-match tables, histograms, and quantiles

## Background references

- Sacchi, R., Menacho, A. H., Seitfudem, G., Agez, M., Schlesinger-Martinat, J., Koyamparambath, A., Saldivar, J. S., Loubet, P., & Bauer, C. (2025). Contextual LCIA without the overhead: an exchange-based framework for flexible impact assessment. *The International Journal of Life Cycle Assessment, 30*(12), 3087-3101. https://doi.org/10.1007/s11367-025-02551-7
- Seitfudem, G., Berger, M., Schmied, H. M., & Boulay, A. (2025). The updated and improved method for water scarcity impact assessment in LCA, AWARE2.0. *Journal of Industrial Ecology, 29*(3), 891-907. https://doi.org/10.1111/jiec.70023


## 1) What changes relative to Day 1?

Day 1 question: "What if the inventory exchanges themselves are uncertain?"

Day 2 question: "What if the characterization factor applied to an exchange varies because the exact regional context is not fully fixed?"


## 2) Inspect two bundled `edges` examples with uncertainty blocks

`lcia_example_5.json` and `lcia_example_6.json` illustrate two styles:

- standard continuous uncertainty distributions
- nested discrete-plus-continuous uncertainty structures


In [ ]:
import json
from pathlib import Path

import bw2data as bd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

from edges import EdgeLCIA

example_5_path = Path('assets/edges_examples/lcia_example_5.json')
example_6_path = Path('assets/edges_examples/lcia_example_6.json')


def summarize_uncertain_method(path):
    data = json.loads(path.read_text())
    return pd.DataFrame([
        {
            'method file': path.name,
            'supplier': exc['supplier']['name'],
            'consumer matrix': exc['consumer']['matrix'],
            'value': exc['value'],
            'uncertainty distribution': exc.get('uncertainty', {}).get('distribution', 'none'),
        }
        for exc in data['exchanges']
    ])

pd.concat([
    summarize_uncertain_method(example_5_path),
    summarize_uncertain_method(example_6_path),
], ignore_index=True)

## 3) Pick one real BAFU activity

We use the BAFU activity `Irrigating` in `CH`. It is a good teaching case because it has:

- direct fossil air emissions that make `lcia_example_5` meaningful;
- a direct `Water, river` resource uptake that makes `lcia_example_6` meaningful too.

In [ ]:
bd.projects.set_current('paris-lca-course-2026')
db = bd.Database('bafu')

activity = [
    act for act in db
    if act['name'] == 'Irrigating'
    and act.get('location') == 'CH'
][0]

{
    'name': activity['name'],
    'location': activity['location'],
    'unit': activity['unit'],
    'reference product': activity['reference product'],
}

In [ ]:
relevant_biosphere_flows = pd.DataFrame([
    {
        'supplier name': exc.input['name'],
        'supplier categories': ' :: '.join(exc.input.get('categories', ())),
        'amount': exc['amount'],
        'unit': exc.input['unit'],
    }
    for exc in activity.biosphere()
    if exc.input['name'] in {'Carbon dioxide, fossil', 'Methane, fossil', 'Nitrogen monoxide'}
    or (
        exc.input['name'].startswith('Water')
        and tuple(exc.input.get('categories', ()))[:2] == ('natural resource', 'in water')
    )
])

relevant_biosphere_flows

## 4) Run the two simple methods deterministically

We first keep the CFs fixed at their nominal `value` and look only at the direct matches on the `Irrigating` activity itself.

In [ ]:
method_paths = {
    'Simple GWP with uncertainty': example_5_path,
    'Simple water scarcity with uncertainty': example_6_path,
}
method_units = {
    label: json.loads(path.read_text())['unit']
    for label, path in method_paths.items()
}

deterministic_results = {}
for label, method_path in method_paths.items():
    lca = EdgeLCIA(
        demand={activity: 1},
        method=('teaching', label),
        filepath=str(method_path),
    )
    lca.lci()
    lca.map_exchanges()
    lca.evaluate_cfs()
    lca.lcia()

    cf_table = lca.generate_cf_table(include_unmatched=True)
    direct_matches = cf_table[
        (cf_table['consumer name'] == activity['name'])
        & (cf_table['consumer reference product'] == activity['reference product'])
        & (cf_table['consumer location'] == activity['location'])
        & (cf_table['CF'].notna())
    ][['supplier name', 'supplier categories', 'amount', 'CF', 'impact']].copy()

    deterministic_results[label] = {
        'lca': lca,
        'direct_matches': direct_matches,
    }

deterministic_summary = pd.DataFrame([
    {
        'method': label,
        'unit': method_units[label],
        'deterministic score': result['lca'].score,
        'direct matched exchanges': len(result['direct_matches']),
    }
    for label, result in deterministic_results.items()
]).set_index('method')

deterministic_summary

## 5) Add CF uncertainty and rerun

Now we keep the same real BAFU inventory, but allow the CFs from the JSON files to vary across repeated `edges` iterations.

In [ ]:
iterations = 1000
stochastic_results = {}

for label, method_path in method_paths.items():
    lca = EdgeLCIA(
        demand={activity: 1},
        method=('teaching', label),
        filepath=str(method_path),
        use_distributions=True,
        iterations=iterations,
    )
    lca.lci()
    lca.map_exchanges()
    lca.evaluate_cfs()
    lca.lcia()

    stochastic_results[label] = {
        'lca': lca,
        'scores': np.asarray(lca.score, dtype=float),
    }

stochastic_summary = pd.DataFrame([
    {
        'method': label,
        'unit': method_units[label],
        'deterministic score': deterministic_results[label]['lca'].score,
        'mean': result['scores'].mean(),
        'p05': np.quantile(result['scores'], 0.05),
        'p50': np.quantile(result['scores'], 0.50),
        'p95': np.quantile(result['scores'], 0.95),
        'coefficient of variation': result['scores'].std(ddof=0) / result['scores'].mean(),
    }
    for label, result in stochastic_results.items()
]).set_index('method')

stochastic_summary

## 6) Visualize the score spread

Because the two methods use different units, we plot each distribution separately and mark the deterministic score with a dashed line.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
colors = {
    'Simple GWP with uncertainty': '#c44e52',
    'Simple water scarcity with uncertainty': '#4c72b0',
}

for ax, label in zip(axes, method_paths):
    scores = stochastic_results[label]['scores']
    ax.hist(scores, bins=30, color=colors[label], alpha=0.85)
    ax.axvline(
        deterministic_results[label]['lca'].score,
        color='black',
        linestyle='--',
        linewidth=1.5,
        label='Deterministic score',
    )
    ax.set_title(label)
    ax.set_xlabel(method_units[label])
    ax.set_ylabel('Iterations')
    ax.legend()

plt.tight_layout()
plt.show()

## 7) Run Monte Carlo with a real regionalized method: `AWARE 2.0`

The previous examples used small local JSON files. We can now switch to a real regionalized method with uncertainty and run the same `Irrigating` activity through `AWARE 2.0`. Before launching `EdgeLCIA`, we first inspect the uncertainty blocks stored in the packaged AWARE JSON file itself.

Start by reading the real method file and building a six-panel overview. The second panel now maps how many uncertain AWARE entries are available for each country, while the other panels show the uncertainty-family mix and one representative `Water, river` example for each distribution.

In [ ]:
import edges
import country_converter as coco
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from edges.uncertainty import sample_cf_distribution

aware_json_path = Path(edges.__file__).resolve().parent / 'data' / 'AWARE 2.0_Country_all_yearly.json'
aware_method_data = json.loads(aware_json_path.read_text())

cc = coco.CountryConverter()
country_code_table = cc.data[['ISO2', 'ISO3']].dropna().drop_duplicates()
iso2_to_iso3 = country_code_table.set_index('ISO2')['ISO3'].astype(str).to_dict()
valid_iso3 = set(country_code_table['ISO3'].astype(str))

location_prefixes = {
    str(exc['consumer']['location']).split('-', 1)[0]
    for exc in aware_method_data['exchanges']
}
location_prefix_to_iso3 = {}
for code in location_prefixes:
    if len(code) == 2:
        location_prefix_to_iso3[code] = iso2_to_iso3.get(code)
    elif len(code) == 3 and code in valid_iso3:
        location_prefix_to_iso3[code] = code
    else:
        location_prefix_to_iso3[code] = None

aware_uncertainty_rows = []
for exc in aware_method_data['exchanges']:
    uncertainty = exc.get('uncertainty')
    if not uncertainty:
        continue
    params = uncertainty.get('parameters', {})
    location = str(exc['consumer']['location'])
    location_prefix = location.split('-', 1)[0]
    aware_uncertainty_rows.append({
        'supplier name': exc['supplier']['name'],
        'location': location,
        'country iso3': location_prefix_to_iso3.get(location_prefix),
        'value': exc['value'],
        'weight': exc.get('weight', np.nan),
        'distribution': uncertainty['distribution'],
        'support size': len(params.get('values', [])) if uncertainty['distribution'] == 'discrete_empirical' else np.nan,
    })

aware_uncertainty_df = pd.DataFrame(aware_uncertainty_rows)
aware_distribution_order = ['discrete_empirical', 'lognorm', 'triang', 'weibull_min']
aware_distribution_colors = {
    'discrete_empirical': '#4c72b0',
    'lognorm': '#55a868',
    'triang': '#c44e52',
    'weibull_min': '#8172b3',
}
aware_country_uncertainty = (
    aware_uncertainty_df.dropna(subset=['country iso3'])
    .groupby('country iso3')
    .size()
    .rename('uncertain CF entries')
    .reset_index()
)

water_river_examples = {}
for distribution in aware_distribution_order:
    water_river_examples[distribution] = next(
        exc
        for exc in aware_method_data['exchanges']
        if exc['supplier']['name'] == 'Water, river'
        and exc.get('uncertainty', {}).get('distribution') == distribution
    )


In [ ]:
aware_uncertainty_overview = make_subplots(
    rows=3,
    cols=2,
    specs=[
        [{'type': 'xy'}, {'type': 'choropleth'}],
        [{'type': 'xy'}, {'type': 'xy'}],
        [{'type': 'xy'}, {'type': 'xy'}],
    ],
    subplot_titles=(
        'a) Uncertainty families across AWARE exchanges',
        'b) Countries covered by uncertain AWARE entries',
        f"c) Discrete empirical example: Water, river in {water_river_examples['discrete_empirical']['consumer']['location']}",
        f"d) Lognormal example: Water, river in {water_river_examples['lognorm']['consumer']['location']}",
        f"e) Triangular example: Water, river in {water_river_examples['triang']['consumer']['location']}",
        f"f) Weibull example: Water, river in {water_river_examples['weibull_min']['consumer']['location']}",
    ),
    vertical_spacing=0.1,
    horizontal_spacing=0.08,
)

# a) Count uncertainty families
dist_counts = aware_uncertainty_df['distribution'].value_counts().reindex(aware_distribution_order)
aware_uncertainty_overview.add_trace(
    go.Bar(
        x=dist_counts.index,
        y=dist_counts.values,
        marker_color=[aware_distribution_colors[d] for d in dist_counts.index],
        showlegend=False,
    ),
    row=1,
    col=1,
)

# b) Country-level map of uncertain entries
aware_uncertainty_overview.add_trace(
    go.Choropleth(
        locations=aware_country_uncertainty['country iso3'],
        z=aware_country_uncertainty['uncertain CF entries'],
        colorscale='YlOrRd',
        marker_line_color='white',
        marker_line_width=0.25,
        colorbar_title='Uncertain<br>entries',
        hovertemplate='%{location}: %{z} uncertain entries<extra></extra>',
    ),
    row=1,
    col=2,
)

# c) Representative discrete empirical example
example = water_river_examples['discrete_empirical']
params = example['uncertainty']['parameters']
values = np.asarray(params['values'], dtype=float)
weights = np.asarray(params['weights'], dtype=float)
mask = weights > 0
aware_uncertainty_overview.add_trace(
    go.Bar(
        x=values[mask],
        y=weights[mask],
        marker_color=aware_distribution_colors['discrete_empirical'],
        showlegend=False,
        hovertemplate='CF %{x:.2f}<br>Weight %{y:.3f}<extra></extra>',
    ),
    row=2,
    col=1,
)
aware_uncertainty_overview.add_trace(
    go.Scatter(
        x=[example['value'], example['value']],
        y=[0, float(weights[mask].max()) * 1.05],
        mode='lines',
        line=dict(color='black', dash='dash', width=1.5),
        showlegend=False,
        hoverinfo='skip',
    ),
    row=2,
    col=1,
)

for row, col, distribution in [(2, 2, 'lognorm'), (3, 1, 'triang'), (3, 2, 'weibull_min')]:
    example = water_river_examples[distribution]
    samples = sample_cf_distribution(
        cf=example,
        n=5000,
        parameters={},
        random_state=np.random.default_rng(42),
        use_distributions=True,
    )
    hist_counts, hist_edges = np.histogram(samples, bins=40, density=True)
    aware_uncertainty_overview.add_trace(
        go.Histogram(
            x=samples,
            nbinsx=40,
            histnorm='probability density',
            marker_color=aware_distribution_colors[distribution],
            opacity=0.85,
            showlegend=False,
            hovertemplate='CF %{x:.2f}<br>Density %{y:.4f}<extra></extra>',
        ),
        row=row,
        col=col,
    )
    aware_uncertainty_overview.add_trace(
        go.Scatter(
            x=[example['value'], example['value']],
            y=[0, float(hist_counts.max()) * 1.05],
            mode='lines',
            line=dict(color='black', dash='dash', width=1.5),
            showlegend=False,
            hoverinfo='skip',
        ),
        row=row,
        col=col,
    )

aware_uncertainty_overview.update_xaxes(title_text='Distribution', tickangle=20, row=1, col=1)
aware_uncertainty_overview.update_yaxes(title_text='Number of uncertain CF entries', row=1, col=1)
aware_uncertainty_overview.update_xaxes(title_text='CF value', row=2, col=1)
aware_uncertainty_overview.update_yaxes(title_text='Probability mass', row=2, col=1)
aware_uncertainty_overview.update_xaxes(title_text='Sampled CF value', row=2, col=2)
aware_uncertainty_overview.update_yaxes(title_text='Density', row=2, col=2)
aware_uncertainty_overview.update_xaxes(title_text='Sampled CF value', row=3, col=1)
aware_uncertainty_overview.update_yaxes(title_text='Density', row=3, col=1)
aware_uncertainty_overview.update_xaxes(title_text='Sampled CF value', row=3, col=2)
aware_uncertainty_overview.update_yaxes(title_text='Density', row=3, col=2)
aware_uncertainty_overview.update_geos(
    showcoastlines=False,
    showcountries=True,
    countrycolor='white',
    showframe=False,
    projection_type='natural earth',
    fitbounds='locations',
    bgcolor='rgba(0,0,0,0)',
    row=1,
    col=2,
)
aware_uncertainty_overview.update_layout(
    height=1050,
    width=980,
    title_text='What the uncertainty data look like in AWARE 2.0',
    title_x=0.5,
    bargap=0.15,
    margin=dict(t=90, l=40, r=20, b=40),
)

aware_uncertainty_overview

In [ ]:
aware_method = ('AWARE 2.0', 'Country', 'all', 'yearly')

aware_lca = EdgeLCIA(
    demand={activity: 1},
    method=aware_method,
)
aware_lca.lci()
aware_lca.map_exchanges()
aware_lca.map_aggregate_locations()
aware_lca.map_dynamic_locations()
aware_lca.map_contained_locations()
aware_lca.map_remaining_locations_to_global()
aware_lca.evaluate_cfs()
aware_lca.lcia()

aware_direct_rows = aware_lca.generate_cf_table(include_unmatched=False)
aware_direct_rows = aware_direct_rows[
    (aware_direct_rows['consumer name'] == activity['name'])
    & (aware_direct_rows['consumer reference product'] == activity['reference product'])
    & (aware_direct_rows['consumer location'] == activity['location'])
    & (aware_direct_rows['CF'].notna())
][['supplier name', 'supplier categories', 'amount', 'CF', 'impact']].copy()

print('Deterministic AWARE score:', aware_lca.score)
aware_direct_rows

In [ ]:
aware_iterations = 1000
aware_mc = EdgeLCIA(
    demand={activity: 1},
    method=aware_method,
    use_distributions=True,
    iterations=aware_iterations,
)
aware_mc.lci()
aware_mc.map_exchanges()
aware_mc.map_aggregate_locations()
aware_mc.map_dynamic_locations()
aware_mc.map_contained_locations()
aware_mc.map_remaining_locations_to_global()
aware_mc.evaluate_cfs()
aware_mc.lcia()

aware_scores = np.asarray(aware_mc.score, dtype=float)
aware_summary = pd.Series({
    'deterministic score': aware_lca.score,
    'mean': aware_scores.mean(),
    'p05': np.quantile(aware_scores, 0.05),
    'p50': np.quantile(aware_scores, 0.50),
    'p95': np.quantile(aware_scores, 0.95),
    'coefficient of variation': aware_scores.std(ddof=0) / aware_scores.mean(),
}, name='AWARE 2.0 Monte Carlo')

aware_summary.to_frame()

In [ ]:
aware_mc_direct_rows = aware_mc.generate_cf_table(include_unmatched=False)
aware_mc_direct_rows = aware_mc_direct_rows[
    (aware_mc_direct_rows['consumer name'] == activity['name'])
    & (aware_mc_direct_rows['consumer reference product'] == activity['reference product'])
    & (aware_mc_direct_rows['consumer location'] == activity['location'])
    & (aware_mc_direct_rows['CF (mean)'].notna())
]

aware_mc_direct_rows

Notice how the water uptake and release values are symmetrically drawn within the same activity.  
The next figure extracts the per-iteration CF draws for the first two rows and makes that symmetry explicit.

In [ ]:
def get_mc_cf_samples_for_row(lca, row):
    cm = lca.characterization_matrix
    supplier_categories = tuple(row['supplier categories'])

    for i, j in zip(*cm.sum(axis=2).nonzero()):
        consumer = bd.get_activity(lca.reversed_activity[j])
        supplier = bd.get_activity(lca.reversed_biosphere[i])

        if (
            supplier['name'] == row['supplier name']
            and tuple(supplier.get('categories', ())) == supplier_categories
            and consumer['name'] == row['consumer name']
            and consumer.get('reference product') == row['consumer reference product']
            and consumer.get('location') == row['consumer location']
        ):
            return np.asarray(cm[i, j, :].todense()).ravel().astype(float)

    raise KeyError(f"Could not recover Monte Carlo CF samples for row: {row['supplier name']}")


water_balance_rows = aware_mc_direct_rows.iloc[:2].copy().reset_index(drop=True)
water_balance_samples = [
    get_mc_cf_samples_for_row(aware_mc, row)
    for _, row in water_balance_rows.iterrows()
]

uptake_samples = water_balance_samples[0]
release_samples = water_balance_samples[1]
iterations_to_show = min(200, aware_iterations)
iteration_index = np.arange(1, iterations_to_show + 1)
mirrored_release = -release_samples
max_abs_difference = np.abs(uptake_samples - mirrored_release).max()

fig, ax = plt.subplots(1, 1, figsize=(10.5, 3.8))

ax.plot(
    iteration_index,
    uptake_samples[:iterations_to_show],
    color='#1f77b4',
    linewidth=1.5,
    label=water_balance_rows.iloc[0]['supplier name'],
)
ax.plot(
    iteration_index,
    release_samples[:iterations_to_show],
    color='#d62728',
    linewidth=1.5,
    label=water_balance_rows.iloc[1]['supplier name'],
)
ax.axhline(0, color='black', linewidth=0.8, alpha=0.6)
ax.set_title('Signed CF draws for the first 200 iterations')
ax.set_xlabel('Iteration')
ax.set_ylabel('CF')
ax.legend(frameon=False)


plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(6.2, 4.0))
ax.hist(aware_scores, bins=30, color='#1f77b4', alpha=0.85)
ax.axvline(
    aware_lca.score,
    color='black',
    linestyle='--',
    linewidth=1.5,
    label='Deterministic AWARE score',
)
ax.axvline(
    aware_summary['mean'],
    color='#d62728',
    linestyle='-',
    linewidth=1.8,
    label='Mean',
)
ax.axvline(
    aware_summary['p05'],
    color='#ff7f0e',
    linestyle=':',
    linewidth=1.8,
    label='5th quantile',
)
ax.axvline(
    aware_summary['p95'],
    color='#ff7f0e',
    linestyle=':',
    linewidth=1.8,
    label='95th quantile',
)
ax.set_title('AWARE 2.0 Monte Carlo on Irrigating (CH)')
ax.set_xlabel('m$^3$ deprived-eq.')
ax.set_ylabel('Iterations')
ax.legend()
plt.tight_layout()
plt.show()

## 8) Combine inventory and characterization uncertainty

So far, the Monte Carlo only varied the characterization factors. `EdgeLCIA` can now also reuse the stochastic `bw2calc` workflow and propagate inventory uncertainty at the same time.

The `AWARE` example above is strongly affected by uncertain water-throughput rows in the background system. To get a cleaner comparison, we can keep the same `Irrigating` activity but switch to a land-occupation method from the packaged `edges` data: `GLAM3 biodiversity occupation average amphibians`.

This method only characterizes occupation flows, so it gives a more stable comparison between:

- uncertainty in the characterization factors only
- joint uncertainty in both the inventory and the characterization factors


In [ ]:
import edges

glam3_method_path = (
    Path(edges.__file__).resolve().parent
    / 'data'
    / 'GLAM3_biodiversity_occupation_average_amphibians.json'
)
glam3_method_unit = json.loads(glam3_method_path.read_text())['unit']
glam3_iterations = 200

glam3_det = EdgeLCIA(
    demand={activity: 1},
    method=('teaching', 'GLAM3 amphibians occupation'),
    filepath=str(glam3_method_path),
)
glam3_det.lci()
glam3_det.map_exchanges()
glam3_det.map_aggregate_locations()
glam3_det.map_dynamic_locations()
glam3_det.map_contained_locations()
glam3_det.map_remaining_locations_to_global()
glam3_det.evaluate_cfs()
glam3_det.lcia()

glam3_cf_only = EdgeLCIA(
    demand={activity: 1},
    method=('teaching', 'GLAM3 amphibians occupation'),
    filepath=str(glam3_method_path),
    use_distributions=True,
    iterations=glam3_iterations,
)
glam3_cf_only.lci()
glam3_cf_only.map_exchanges()
glam3_cf_only.map_aggregate_locations()
glam3_cf_only.map_dynamic_locations()
glam3_cf_only.map_contained_locations()
glam3_cf_only.map_remaining_locations_to_global()
glam3_cf_only.evaluate_cfs()
glam3_cf_only.lcia()

glam3_joint = EdgeLCIA(
    demand={activity: 1},
    method=('teaching', 'GLAM3 amphibians occupation'),
    filepath=str(glam3_method_path),
    use_distributions=True,
    inventory_use_distributions=True,
    store_inventory_samples=True,
    iterations=glam3_iterations,
)
glam3_joint.lci()
glam3_joint.map_exchanges()
glam3_joint.map_aggregate_locations()
glam3_joint.map_dynamic_locations()
glam3_joint.map_contained_locations()
glam3_joint.map_remaining_locations_to_global()
glam3_joint.evaluate_cfs()
glam3_joint.lcia()

glam3_cf_only_scores = np.asarray(glam3_cf_only.score, dtype=float)
glam3_joint_scores = np.asarray(glam3_joint.score, dtype=float)

glam3_uncertainty_comparison = pd.DataFrame([
    {
        'mode': 'CF uncertainty only',
        'iterations': glam3_iterations,
        'deterministic score': glam3_det.score,
        'mean': glam3_cf_only_scores.mean(),
        'p05': np.quantile(glam3_cf_only_scores, 0.05),
        'p50': np.quantile(glam3_cf_only_scores, 0.50),
        'p95': np.quantile(glam3_cf_only_scores, 0.95),
        'negative share': (glam3_cf_only_scores < 0).mean(),
    },
    {
        'mode': 'Joint inventory + CF uncertainty',
        'iterations': glam3_iterations,
        'deterministic score': glam3_det.score,
        'mean': glam3_joint_scores.mean(),
        'p05': np.quantile(glam3_joint_scores, 0.05),
        'p50': np.quantile(glam3_joint_scores, 0.50),
        'p95': np.quantile(glam3_joint_scores, 0.95),
        'negative share': (glam3_joint_scores < 0).mean(),
    },
]).set_index('mode')

display(glam3_uncertainty_comparison)

glam3_joint_direct_rows = glam3_joint.generate_cf_table(include_unmatched=False)
glam3_joint_direct_rows = glam3_joint_direct_rows[
    (glam3_joint_direct_rows['consumer name'] == activity['name'])
    & (glam3_joint_direct_rows['consumer reference product'] == activity['reference product'])
    & (glam3_joint_direct_rows['consumer location'] == activity['location'])
    & (glam3_joint_direct_rows['CF (mean)'].notna())
][[
    'supplier name',
    'supplier categories',
    'amount (mean)',
    'amount (5th)',
    'amount (95th)',
    'CF (mean)',
    'impact (mean)',
]].copy()

glam3_joint_direct_rows


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.2))
glam3_colors = {
    'CF uncertainty only': '#4c72b0',
    'Joint inventory + CF uncertainty': '#55a868',
}
glam3_scores_to_plot = {
    'CF uncertainty only': glam3_cf_only_scores,
    'Joint inventory + CF uncertainty': glam3_joint_scores,
}

for ax, (label, scores) in zip(axes, glam3_scores_to_plot.items()):
    ax.hist(scores, bins=25, color=glam3_colors[label], alpha=0.85)
    ax.axvline(glam3_det.score, color='black', linestyle='--', linewidth=1.5, label='Deterministic score')
    ax.axvline(scores.mean(), color='black', linewidth=1.5, label='Mean')
    ax.axvline(np.quantile(scores, 0.05), color='#ff7f0e', linestyle=':', linewidth=2, label='5th / 95th quantiles')
    ax.axvline(np.quantile(scores, 0.95), color='#ff7f0e', linestyle=':', linewidth=2)
    ax.set_title(label)
    ax.set_xlabel(glam3_method_unit)
    ax.set_ylabel('Iterations')
    ax.legend(frameon=False)

plt.tight_layout()
plt.show()


With this occupation-only `GLAM3` method, the joint run is wider than the CF-only run.

## Key caution for practitioners

Even with one fixed real activity, uncertain or spatially variable CFs can produce a wide spread of LCIA scores. Once you also propagate inventory uncertainty, the spread can widen substantially and even change the sign of scores, but the magnitude of that effect depends strongly on which exchanges the chosen method actually characterizes.


## Recap

After this notebook, you should now be able to:

- explain the difference between inventory uncertainty and CF variability
- inspect uncertainty information in local `edges` method files
- run uncertain `EdgeLCIA` calculations on a real BAFU activity
- run a Monte Carlo with a real regionalized method such as `AWARE 2.0`
- compare CF-only uncertainty against a joint inventory + characterization Monte Carlo
- contrast that comparison across two real methods with different coverage, such as `AWARE` and `GLAM3`
- connect direct matched exchanges to the resulting score distributions
- summarize score spread with histograms and quantiles
